<table style="border: none; border-collapse: collapse;">
    <tr>
        <td style="width: 20%; vertical-align: middle; padding-right: 10px;">
            <img src="https://i.imgur.com/nt7hloA.png" width="100">
        </td>
        <td style="width: 2px; text-align: center;">
            <font color="#0030A1" size="7">|</font><br>
            <font color="#0030A1" size="7">|</font>
        </td>
        <td>
            <p style="font-variant: small-caps;"><font color="#0030A1" size="5">
                <b>Facultad de Ciencias Exactas, Naturales y Ambientales</b>
            </font> </p>
            <p style="font-variant: small-caps;"><font color="#0030A1" size="4">
                Ciencias de Datos para el Estudio de Casos &bull; Kevin Rojas Satián &bull; 13 de mayo del 2026 Taller de implementación 2
            </font></p>
            <p style="font-style: oblique;"><font color="#0030A1" size="3">
                Alanis Cristhine Caicedo A. &bull; 13 de mayo del 2026
            </font></p>
        </td>  
    </tr>
</table>

In [3]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import os
import re
import zipfile
import sqlite3
from pathlib import Path

import pandas as pd
import numpy as np

In [4]:
# ============================================================
# 1. CONFIGURACIÓN
# ============================================================

# Cambia esta ruta si el ZIP está en Google Drive
ZIP_PATH = "/content/2_BDD_DATOS_ABIERTOS_ENEMDU_2025_CSV.zip"

# Carpeta donde se extraerán los archivos
EXTRACT_DIR = Path("/content/enemdu_2025_extraido")

# Nombre de la base SQLite final
DB_PATH = "/content/enemdu_2025.sqlite"

# Tamaño de lectura por bloques
CHUNKSIZE = 50000


In [5]:
# ============================================================
# 2. EXTRAER ZIP
# ============================================================

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Archivos extraídos:")
for f in EXTRACT_DIR.iterdir():
    print("-", f.name)


Archivos extraídos:
- Diccionario de Datos_persona_anual_2025.xlsx
- Metadatos_vivienda_hogar_anual_2025.xlsx
- Metadatos_persona_anual_2025.xlsx
- BDDenemdu_personas_2025_anual.csv
- Diccionario de Datos_vivienda_anual_2025.xlsx
- BDDenemdu_vivienda_2025_anual.csv


In [6]:
# ============================================================
# 3. LOCALIZAR ARCHIVOS PRINCIPALES
# ============================================================

personas_csv = EXTRACT_DIR / "BDDenemdu_personas_2025_anual.csv"
vivienda_csv = EXTRACT_DIR / "BDDenemdu_vivienda_2025_anual.csv"

dicc_persona_xlsx = EXTRACT_DIR / "Diccionario de Datos_persona_anual_2025.xlsx"
dicc_vivienda_xlsx = EXTRACT_DIR / "Diccionario de Datos_vivienda_anual_2025.xlsx"
meta_persona_xlsx = EXTRACT_DIR / "Metadatos_persona_anual_2025.xlsx"
meta_vivienda_xlsx = EXTRACT_DIR / "Metadatos_vivienda_hogar_anual_2025.xlsx"

assert personas_csv.exists(), "No se encontró el CSV de personas"
assert vivienda_csv.exists(), "No se encontró el CSV de vivienda"

print("\nCSV encontrados correctamente.")



CSV encontrados correctamente.


In [7]:
# ============================================================
# 4. FUNCIONES DE LIMPIEZA / ETL
# ============================================================

def limpiar_nombre_columna(col):
    """
    Normaliza nombres de columnas para SQLite.
    """
    col = str(col).strip().lower()
    col = re.sub(r"[^\w]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col


def limpiar_dataframe(df):
    """
    Limpieza general:
    - normaliza nombres de columnas
    - elimina espacios
    - convierte blancos a NULL
    """
    df.columns = [limpiar_nombre_columna(c) for c in df.columns]

    for col in df.columns:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace({
            "": pd.NA,
            " ": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "NULL": pd.NA,
            "null": pd.NA
        })

    return df


def normalizar_claves_enemdu(df):
    """
    Normaliza claves importantes de ENEMDU.
    Se conservan como texto para no perder ceros iniciales.
    """

    columnas_texto_con_ceros = [
        "ciudad",
        "conglomerado",
        "panelm",
        "vivienda",
        "hogar",
        "upm",
        "id_vivienda",
        "id_hogar",
        "id_persona"
    ]

    for col in columnas_texto_con_ceros:
        if col in df.columns:
            df[col] = df[col].astype("string").str.strip()

    # Ciudad en personas puede venir como 10150 y en vivienda como 010150.
    # Se normaliza a 6 dígitos.
    if "ciudad" in df.columns:
        df["ciudad_key"] = (
            df["ciudad"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(6)
        )

    # Provincia se normaliza a 2 dígitos cuando exista
    if "prov" in df.columns:
        df["prov_key"] = (
            df["prov"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(2)
        )

    # Periodo y mes
    if "periodo" in df.columns:
        df["periodo_key"] = df["periodo"].astype("string").str.strip()

    if "mes" in df.columns:
        df["mes_key"] = (
            df["mes"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.zfill(2)
        )

    return df


def convertir_decimal_coma(serie):
    """
    Convierte números con coma decimal a float.
    Ejemplo: '14,840923259728' -> 14.840923259728
    """
    return (
        serie
        .astype("string")
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )


def convertir_tipos_principales(df):
    """
    Convierte variables numéricas principales.
    No convierte todos los códigos para no dañar variables categóricas.
    """
    columnas_float = [
        "fexp",
        "ingrl",
        "ingpc"
    ]

    columnas_int = [
        "periodo",
        "mes",
        "area",
        "pobreza",
        "epobreza",
        "condact",
        "empleo",
        "desempleo",
        "secemp",
        "grupo1",
        "rama1",
        "dominio"
    ]

    for col in columnas_float:
        if col in df.columns:
            df[col] = convertir_decimal_coma(df[col])

    for col in columnas_int:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    return df


def preparar_chunk(df):
    """
    Aplica todo el ETL a un bloque de datos.
    """
    df = limpiar_dataframe(df)
    df = normalizar_claves_enemdu(df)
    df = convertir_tipos_principales(df)
    return df


def leer_csv_chunks(path):
    """
    Lee CSV ENEMDU por bloques.
    El archivo usa separador ; y encoding UTF-8-SIG.
    """
    return pd.read_csv(
        path,
        sep=";",
        encoding="utf-8-sig",
        dtype=str,
        chunksize=CHUNKSIZE,
        keep_default_na=False
    )

In [8]:
# ============================================================
# 5. CREAR BASE SQLITE
# ============================================================

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")
cursor.execute("PRAGMA journal_mode = WAL;")
cursor.execute("PRAGMA synchronous = NORMAL;")

print("\nBase SQLite creada:", DB_PATH)



Base SQLite creada: /content/enemdu_2025.sqlite


In [9]:
# ============================================================
# 6. CARGAR STAGING Y FACT TABLES
# ============================================================

def cargar_csv_a_sqlite(csv_path, tabla_staging, tabla_fact):
    """
    Carga datos en:
    - tabla staging
    - tabla fact limpia
    """
    first = True
    total = 0

    for chunk in leer_csv_chunks(csv_path):
        # Staging: limpieza mínima
        stg = limpiar_dataframe(chunk.copy())

        stg.to_sql(
            tabla_staging,
            conn,
            if_exists="replace" if first else "append",
            index=False
        )

        # Fact: ETL completo
        fact = preparar_chunk(chunk.copy())

        fact.to_sql(
            tabla_fact,
            conn,
            if_exists="replace" if first else "append",
            index=False
        )

        total += len(chunk)
        first = False
        print(f"{tabla_fact}: {total:,} filas cargadas")

    print(f"Finalizado: {tabla_fact} con {total:,} filas\n")


cargar_csv_a_sqlite(personas_csv, "stg_personas_raw", "fact_persona")
cargar_csv_a_sqlite(vivienda_csv, "stg_vivienda_raw", "fact_vivienda")

fact_persona: 50,000 filas cargadas
fact_persona: 100,000 filas cargadas
fact_persona: 150,000 filas cargadas
fact_persona: 200,000 filas cargadas
fact_persona: 250,000 filas cargadas
fact_persona: 300,000 filas cargadas
fact_persona: 334,786 filas cargadas
Finalizado: fact_persona con 334,786 filas

fact_vivienda: 50,000 filas cargadas
fact_vivienda: 100,000 filas cargadas
fact_vivienda: 104,738 filas cargadas
Finalizado: fact_vivienda con 104,738 filas



In [10]:
# ============================================================
# 7. CREAR DIMENSIONES
# ============================================================

# ----------------------------
# DIMENSIÓN PERIODO
# ----------------------------

cursor.execute("DROP TABLE IF EXISTS dim_periodo;")

cursor.execute("""
CREATE TABLE dim_periodo AS
SELECT DISTINCT
    periodo_key,
    CAST(SUBSTR(periodo_key, 1, 4) AS INTEGER) AS anio,
    CAST(SUBSTR(periodo_key, 5, 2) AS INTEGER) AS mes_numero,
    mes_key
FROM fact_persona
WHERE periodo_key IS NOT NULL;
""")

cursor.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS idx_dim_periodo_periodo_key
ON dim_periodo(periodo_key);
""")


# ----------------------------
# DIMENSIÓN GEOGRAFÍA
# ----------------------------

cursor.execute("DROP TABLE IF EXISTS dim_geografia;")

cursor.execute("""
CREATE TABLE dim_geografia AS
SELECT DISTINCT
    ciudad_key,
    prov_key,
    area,
    dominio
FROM fact_persona
WHERE ciudad_key IS NOT NULL;
""")

cursor.execute("""
CREATE INDEX IF NOT EXISTS idx_dim_geografia_ciudad_key
ON dim_geografia(ciudad_key);
""")

In [11]:
# ============================================================
# 8. ÍNDICES PARA RELACIONES Y RENDIMIENTO
# ============================================================

indices = [
    "CREATE UNIQUE INDEX IF NOT EXISTS idx_fact_persona_id_persona ON fact_persona(id_persona);",
    "CREATE INDEX IF NOT EXISTS idx_fact_persona_id_hogar ON fact_persona(id_hogar);",
    "CREATE INDEX IF NOT EXISTS idx_fact_persona_id_vivienda ON fact_persona(id_vivienda);",
    "CREATE INDEX IF NOT EXISTS idx_fact_persona_periodo ON fact_persona(periodo_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_persona_ciudad ON fact_persona(ciudad_key);",

    "CREATE UNIQUE INDEX IF NOT EXISTS idx_fact_vivienda_id_hogar ON fact_vivienda(id_hogar);",
    "CREATE INDEX IF NOT EXISTS idx_fact_vivienda_id_vivienda ON fact_vivienda(id_vivienda);",
    "CREATE INDEX IF NOT EXISTS idx_fact_vivienda_periodo ON fact_vivienda(periodo_key);",
    "CREATE INDEX IF NOT EXISTS idx_fact_vivienda_ciudad ON fact_vivienda(ciudad_key);"
]

for sql in indices:
    cursor.execute(sql)

conn.commit()

print("Dimensiones e índices creados correctamente.")

Dimensiones e índices creados correctamente.


In [12]:
# ============================================================
# 9. CARGAR DICCIONARIOS Y METADATOS
# ============================================================

def cargar_excel_a_sqlite(path, prefijo_tabla):
    """
    Carga todas las hojas de un Excel como tablas SQLite.
    """
    if not path.exists():
        print("No existe:", path.name)
        return

    hojas = pd.read_excel(path, sheet_name=None, dtype=str)

    for nombre_hoja, df in hojas.items():
        df = limpiar_dataframe(df)

        tabla = f"{prefijo_tabla}_{limpiar_nombre_columna(nombre_hoja)}"

        df.to_sql(
            tabla,
            conn,
            if_exists="replace",
            index=False
        )

        print(f"Cargado {path.name} -> {tabla}")


cargar_excel_a_sqlite(dicc_persona_xlsx, "dicc_persona")
cargar_excel_a_sqlite(dicc_vivienda_xlsx, "dicc_vivienda")
cargar_excel_a_sqlite(meta_persona_xlsx, "meta_persona")
cargar_excel_a_sqlite(meta_vivienda_xlsx, "meta_vivienda")

Cargado Diccionario de Datos_persona_anual_2025.xlsx -> dicc_persona_hoja1
Cargado Diccionario de Datos_vivienda_anual_2025.xlsx -> dicc_vivienda_hoja1
Cargado Metadatos_persona_anual_2025.xlsx -> meta_persona_hoja1
Cargado Metadatos_vivienda_hogar_anual_2025.xlsx -> meta_vivienda_hoja1


In [13]:
# ============================================================
# 10. VALIDACIONES DEL ETL
# ============================================================

def consulta(sql):
    return pd.read_sql_query(sql, conn)


print("\nResumen de filas:")
display(consulta("""
SELECT 'fact_persona' AS tabla, COUNT(*) AS filas FROM fact_persona
UNION ALL
SELECT 'fact_vivienda' AS tabla, COUNT(*) AS filas FROM fact_vivienda
UNION ALL
SELECT 'dim_periodo' AS tabla, COUNT(*) AS filas FROM dim_periodo
UNION ALL
SELECT 'dim_geografia' AS tabla, COUNT(*) AS filas FROM dim_geografia;
"""))

print("\nDuplicados en claves principales:")
display(consulta("""
SELECT
    (SELECT COUNT(*) FROM fact_persona) AS total_personas,
    (SELECT COUNT(DISTINCT id_persona) FROM fact_persona) AS personas_unicas,
    (SELECT COUNT(*) FROM fact_vivienda) AS total_hogares,
    (SELECT COUNT(DISTINCT id_hogar) FROM fact_vivienda) AS hogares_unicos;
"""))

print("\nPersonas sin hogar relacionado en vivienda:")
display(consulta("""
SELECT COUNT(*) AS personas_sin_hogar_relacionado
FROM fact_persona p
LEFT JOIN fact_vivienda v
    ON p.id_hogar = v.id_hogar
WHERE v.id_hogar IS NULL;
"""))

print("\nMuestra de relación persona - vivienda:")
display(consulta("""
SELECT
    p.id_persona,
    p.id_hogar,
    p.periodo_key,
    p.ciudad_key,
    p.p02 AS sexo_codigo,
    p.p03 AS edad,
    p.ingrl AS ingreso_laboral,
    v.vi01,
    v.vi02,
    v.fexp AS factor_expansion_hogar
FROM fact_persona p
LEFT JOIN fact_vivienda v
    ON p.id_hogar = v.id_hogar
LIMIT 10;
"""))


Resumen de filas:


,tabla,filas
0,fact_persona,334786
1,fact_vivienda,104738
2,dim_periodo,12
3,dim_geografia,933



Duplicados en claves principales:


,total_personas,personas_unicas,total_hogares,hogares_unicos
0,334786,334786,104738,104738



Personas sin hogar relacionado en vivienda:


,personas_sin_hogar_relacionado
0,0



Muestra de relación persona - vivienda:


,id_persona,id_hogar,periodo_key,ciudad_key,sexo_codigo,edad,ingreso_laboral,vi01,vi02,factor_expansion_hogar
0,0101500002040670110106,01015000020406701106,202506,010150,2,64,200.0,1,2,14.840923
1,0101500002040670110109,01015000020406701109,202509,010150,2,65,300.0,1,2,11.559104
2,0101500002040670210109,01015000020406702109,202509,010150,2,66,200.0,1,2,11.559104
3,0101500002040670210209,01015000020406702109,202509,010150,1,66,NaN,1,2,11.559104
4,0101500002040670210309,01015000020406702109,202509,010150,1,45,600.0,1,2,11.559104
5,0101500002040670210409,01015000020406702109,202509,010150,2,24,1004.0,1,2,11.559104
6,0101500002040670310106,01015000020406703106,202506,010150,2,46,880.0,1,1,14.840923
7,0101500002040670310206,01015000020406703106,202506,010150,2,15,NaN,1,1,14.840923
8,0101500002040670310306,01015000020406703106,202506,010150,1,7,NaN,1,1,14.840923
9,0101500002040670310109,01015000020406703109,202509,010150,2,47,440.0,1,1,11.559104


In [14]:
# ============================================================
# 11. CERRAR CONEXIÓN
# ============================================================

conn.commit()
conn.close()

print("\nETL terminado.")
print("Base SQLite creada en:", DB_PATH)


ETL terminado.
Base SQLite creada en: /content/enemdu_2025.sqlite


In [15]:
# ============================================================
# 12. VERIFICACIÓN Y VALIDACIONES
# ============================================================

import sqlite3
import pandas as pd

DB_PATH = "/content/enemdu_2025.sqlite"
conn = sqlite3.connect(DB_PATH)

def consulta(sql):
    return pd.read_sql_query(sql, conn)

# 1. Verificar periodos cargados
display(consulta("""
SELECT periodo_key, COUNT(*) AS total_personas
FROM fact_persona
GROUP BY periodo_key
ORDER BY periodo_key;
"""))

# 2. Verificar meses cargados
display(consulta("""
SELECT mes_key, COUNT(*) AS total_personas
FROM fact_persona
GROUP BY mes_key
ORDER BY mes_key;
"""))

# 3. Revisar edades mínimas y máximas
display(consulta("""
SELECT
    MIN(CAST(p03 AS INTEGER)) AS edad_minima,
    MAX(CAST(p03 AS INTEGER)) AS edad_maxima,
    AVG(CAST(p03 AS INTEGER)) AS edad_promedio
FROM fact_persona
WHERE p03 IS NOT NULL;
"""))

# 4. Revisar ingresos laborales
display(consulta("""
SELECT
    COUNT(*) AS total_registros,
    COUNT(ingrl) AS registros_con_ingreso,
    MIN(ingrl) AS ingreso_minimo,
    MAX(ingrl) AS ingreso_maximo,
    AVG(ingrl) AS ingreso_promedio
FROM fact_persona;
"""))

# 5. Verificar sexo
display(consulta("""
SELECT
    p02 AS sexo_codigo,
    COUNT(*) AS total
FROM fact_persona
GROUP BY p02
ORDER BY p02;
"""))

# 6. Verificar área
display(consulta("""
SELECT
    area,
    COUNT(*) AS total
FROM fact_persona
GROUP BY area
ORDER BY area;
"""))

conn.close()

,periodo_key,total_personas
0,202501,28320
1,202502,28303
2,202503,27932
3,202504,28092
4,202505,27903
5,202506,28072
6,202507,27509
7,202508,27580
8,202509,27332
9,202510,27809


,mes_key,total_personas
0,01,28320
1,02,28303
2,03,27932
3,04,28092
4,05,27903
5,06,28072
6,07,27509
7,08,27580
8,09,27332
9,10,27809


,edad_minima,edad_maxima,edad_promedio
0,0,98,36.374583


,total_registros,registros_con_ingreso,ingreso_minimo,ingreso_maximo,ingreso_promedio
0,334786,146136,-1,999999,3840.422812


,sexo_codigo,total
0,1,161225
1,2,173561


,area,total
0,1,245323
1,2,89463


In [16]:
# ============================================================
# 13. GENERAR ARCHIVO .DB Y .ZIP FINAL PARA DESCARGA
# ============================================================

import shutil
import zipfile
from google.colab import files

DB_PATH = "/content/enemdu_2025.sqlite"
DB_FINAL_PATH = "/content/enemdu_2025.db"
ZIP_FINAL_PATH = "/content/enemdu_2025_db.zip"

# Copiar SQLite como .db
shutil.copy(DB_PATH, DB_FINAL_PATH)

# Crear ZIP con el .db
with zipfile.ZipFile(ZIP_FINAL_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(DB_FINAL_PATH, arcname="enemdu_2025.db")

print("Archivos generados correctamente:")
print(DB_FINAL_PATH)
print(ZIP_FINAL_PATH)

# Descargar ZIP final
files.download(ZIP_FINAL_PATH)

Archivos generados correctamente:
/content/enemdu_2025.db
/content/enemdu_2025_db.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>